In [1]:
%load_ext autoreload
%autoreload 2

import pickle
import ujson
import json
import sys
import os
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import random
import faiss

from tqdm import tqdm
from collections import defaultdict
from typing import Optional

from bioel.utils.umls_utils import UmlsMappings
from bioel.utils.bigbio_utils import CUIS_TO_REMAP, CUIS_TO_EXCLUDE, DATASET_NAMES, VALIDATION_DOCUMENT_IDS
from bioel.utils.bigbio_utils import load_bigbio_dataset, add_deabbreviations, load_dataset_df, dataset_to_documents, dataset_to_df, load_dataset_df, resolve_abbreviation, dataset_unique_tax_ids
from bioel.utils.solve_abbreviation.solve_abbreviations import create_abbrev

from bioel.ontology import BiomedicalOntology
from bioel.models.arboel.biencoder.data.data_utils import process_ontology
from bioel.evaluate import Evaluate

from torch.utils.data import DataLoader

from peft import PeftModel
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from ids import open_ai_api_key

/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING 09-22 11:13:12 cuda.py:22] You are using a deprecated `pynvml` package. Please install `nvidia-ml-py` instead, and make sure to uninstall `pynvml`. When both of them are installed, `pynvml` will take precedence and cause errors. See https://pypi.org/project/pynvml for more information.
WARNING 09-22 11:13:12 cuda.py:69] Detected different devices in the system: 
WARNING 09-22 11:13:12 cuda.py:69] Tesla V100-PCIE-32GB
WARNING 09-22 11:13:12 cuda.py:69] NVIDIA A40
WARNING 09-22 11:13:12 cuda.py:69] NVIDIA A40
WARNING 09-22 11:13:12 cuda.py:69] NVIDIA A40
WARNING 09-22 11:13:12 cuda.py:69] Please make sure to set `CUDA_DEVICE_ORDER=PCI_BUS_ID` to avoid unexpected behavior.


2024-09-22 11:13:12,881	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
from utils_functions import *

In [3]:
import openai
openai.api_key = open_ai_api_key
import re
import ujson
import logging
from collections import Counter, defaultdict
import pandas as pd

# Set up logging configuration
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logging.getLogger("httpx").setLevel(logging.WARNING)
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1" 
sampling_params = SamplingParams(temperature=0, top_p=0.9, max_tokens=1000, stop=["<|eot_id|>"])

In [4]:
# Check how many GPUs are visible
print(f"Visible GPUs: {torch.cuda.device_count()}")

# Check if you can access both GPUs
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    
device_1 = torch.device("cuda:0")  # First GPU (GPU 0)
device_2 = torch.device("cuda:1")  # Second GPU (GPU 1)

seed = 40
np.random.seed(seed)
random.seed(seed)

Visible GPUs: 2
GPU 0: NVIDIA A40
GPU 1: NVIDIA A40


In [5]:
ontology_dir = "/mitchell/entity-linking/kbs/medic.tsv"
name = "medic"
ontology2 = BiomedicalOntology.load_medic(filepath=ontology_dir, name=name)

# entrez_dict = {"name" : "entrez",
#              "filepath" : "/mitchell/entity-linking/el-robustness-comparison/data/gene_info.tsv",
#              "dataset" : "gnormplus",}
# ontology = BiomedicalOntology.load_entrez(**entrez_dict)


[2024-09-22 11:13:13] [ontology.py] [INFO] Reading entrez from /mitchell/entity-linking/kbs/medic.tsv


In [6]:
dataset_name = 'ncbi_disease'
# dataset_name = 'gnormplus'
path_to_abbrev = "/home2/cye73/data_test2/abbreviations.json"
dataset = load_bigbio_dataset(dataset_name)
dataset = add_deabbreviations(dataset, path_to_abbrev)

In [7]:
dataset_df = dataset_to_df(dataset)
test_df = dataset_df[dataset_df['split'] == 'test']
train_df = dataset_df[dataset_df['split'] == 'train']
test_df

,document_id,offsets,text,type,db_ids,split,deabbreviated_text,mention_id
5939,9288106,"[[40, 61]]",ataxia-telangiectasia,[Modifier],[MESH:D001260],test,ataxia-telangiectasia,9288106.1
5948,9288106,"[[72, 97]]",sporadic T-cell leukaemia,[SpecificDisease],[MESH:D015458],test,sporadic T-cell leukaemia,9288106.2
5949,9288106,"[[99, 120]]",Ataxia-telangiectasia,[SpecificDisease],[MESH:D001260],test,Ataxia-telangiectasia,9288106.3
5925,9288106,"[[122, 125]]",A-T,[SpecificDisease],[MESH:D001260],test,Ataxia-telangiectasia,9288106.4
5928,9288106,"[[132, 163]]",recessive multi-system disorder,[DiseaseClass],[MESH:D030342],test,recessive multi-system disorder,9288106.5
...,...,...,...,...,...,...,...,...
6880,9988281,"[[996, 1015]]",breast malignancies,[SpecificDisease],[MESH:D001943],test,breast malignancies,9988281.7
6870,9988281,"[[1123, 1147]]",invasive lobular cancers,[DiseaseClass],[MESH:D018275],test,invasive lobular cancers,9988281.8
6871,9988281,"[[1152, 1179]]",low-grade ductal carcinomas,[SpecificDisease],[MESH:D044584],test,low-grade ductal carcinomas,9988281.9
6873,9988281,"[[1269, 1286]]",ductal carcinomas,[SpecificDisease],[MESH:D044584],test,ductal carcinomas,9988281.10


In [8]:
docs = dataset_to_documents(dataset)
# docs

In [9]:
add_full_context(df = test_df, docs=docs)
add_full_context(df = train_df, docs=docs)
# test_df

/home2/cye73/llm_disambiguator/experiments/utils_functions.py:276: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["contextualized_mention"] = contextualized_mentions
/home2/cye73/llm_disambiguator/experiments/utils_functions.py:276: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["contextualized_mention"] = contextualized_mentions


In [10]:
_, TestMap_mention2context = add_context(df = test_df, docs = docs)
corpus, TrainMap_mention2context = add_context(df = train_df, docs = docs)
_, _ = add_context(df = dataset_df, docs = docs)
# dataset_df

/home2/cye73/llm_disambiguator/experiments/utils_functions.py:322: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["limited_contextualized_mention"] = limited_contextualized_mentions
/home2/cye73/llm_disambiguator/experiments/utils_functions.py:322: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["limited_contextualized_mention"] = limited_contextualized_mentions


In [11]:
# TrainMap_mention2context
# print(len(corpus))

In [12]:
# # # Load a pre-trained SentenceBERT model
# # model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# # # Generate embeddings for the corpus
# # corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer('princeton-nlp/sup-simcse-bert-base-uncased')
# model.to(device_1)
# # Generate embeddings for the corpus
# corpus_embeddings = model.encode(corpus, convert_to_tensor=True)

# corpus_embeddings = corpus_embeddings.cpu().detach().numpy()

In [13]:
# embedding_dimension = corpus_embeddings.shape[1]

# # Create the HNSW index with the correct arguments
# M = 32  # Number of neighbors in the HNSW graph
# index = faiss.IndexHNSWFlat(embedding_dimension, M)

# # Normalize the corpus embeddings if using cosine similarity
# faiss.normalize_L2(corpus_embeddings)

# # Add the embeddings to the index
# index.add(corpus_embeddings)

# # Print the number of sentences added to the index
# print(f"Number of sentences in the index: {index.ntotal}")

In [14]:
# def knn_query(model, index, query, k=5):
#     ''' 
#     Find the top k most similar embeddings of the query from the corpus.
#     ------
#     model : SentenceTransformer model
#     index : faiss index
#     query : str (mention + surrounding context)
#     k : int (number of similar embeddings to find)
#     '''
#     # Generate embedding for the query
#     query_embedding = model.encode(query, convert_to_tensor=True).cpu().detach().numpy()
#     query_embedding = query_embedding.reshape(1, -1)
#     print(query_embedding.shape)
#     # Normalize the query embedding for cosine similarity
#     faiss.normalize_L2(query_embedding)
    
#     # Perform the search
#     distances, indices = index.search(query_embedding, k)
    
#     # print("Query:", TestMap_cui2context[query])
#     # print("\nTop 3 most similar sentences in the corpus:")
    
#     # for i, idxs in enumerate(indices[0]):
#     #     print(f"{i+1}. {corpus[idxs]} (Distance: {distances[0][i]})")
    
#     return indices[0]
    

In [15]:
TrainMap_context2mention = {v: k for k, v in TrainMap_mention2context.items()}
# TrainMap_context2mention

# Load arboel results

In [16]:
dataset_names = ["ncbi_disease"]
model_names = ["arboel_biencoder", "arboel_crossencoder"]
path_to_result = {"ncbi_disease": {
        "arboel_biencoder": "/home2/cye73/results2/arboel/ncbi_disease/biencoder_output_eval.json",
        "arboel_crossencoder": "/home2/cye73/results2/arboel/ncbi_disease/crossencoder_output_eval.json"
    }}

# dataset_names = ["gnormplus"]
# model_names = ["arboel_biencoder", "arboel_crossencoder"]
# path_to_result = {"gnormplus": {
#         "arboel_biencoder": "/home2/cye73/results2/arboel/gnormplus/biencoder_output_eval.json",
#         "arboel_crossencoder": "/home2/cye73/results2/arboel/gnormplus/crossencoder_output_eval.json"
#     }}

abbreviations_path = "/home2/cye73/data_test/abbreviations.json"

evaluator = Evaluate(dataset_names, model_names, path_to_result,
                     abbreviations_path
                     )
evaluator.load_results()
evaluator.process_datasets()
evaluator.evaluate(
    eval_strategies=['basic']
                   )
# evaluator.plot_results(
#     # eval_strategies=['basic']
#     )

dataset : ncbi_disease, model : arboel_biencoder
dataset : ncbi_disease, model : arboel_crossencoder


  0%|          | 0/1 [00:00<?, ?it/s]

Eval Strategy: basic
ncbi_disease


/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:249: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: min_hit_index(x[0], x[1], eval_mode=eval_mode), axis=1
/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:249: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  lambda x: min_hit_index(x[0], x[1], eval_mode=eval_mode), axis=1
/home2/cye73/biomedical-entity-linking/bioel/bioel/evaluate.py:249: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value

In [23]:
number_candidates = 64

# results = evaluator.full_results["basic"]["gnormplus"]
results = evaluator.full_results["basic"]["ncbi_disease"]
results
cols = ['document_id', 'offsets', 'deabbreviated_text', 'db_ids', 'mention_id', 'joined_offsets', 'arboel_biencoder_resolve_abbrev', 'arboel_biencoder_resolve_abbrev_min_hit_index', 'arboel_crossencoder_resolve_abbrev', 'arboel_crossencoder_resolve_abbrev_min_hit_index']
filtered_results = results[cols].rename(columns={'arboel_biencoder_resolve_abbrev': 'biencoder_candidates',
                                                 'arboel_crossencoder_resolve_abbrev': 'crossencoder_candidates',
                                                 'arboel_biencoder_resolve_abbrev_min_hit_index': 'biencoder_hit_index',
                                                 'arboel_crossencoder_resolve_abbrev_min_hit_index': 'crossencoder_hit_index'})

filtered_results = filtered_results[filtered_results['crossencoder_hit_index'] < number_candidates]
# filtered_results = filtered_results[filtered_results['crossencoder_hit_index'] < 50]

# filtered_results = filtered_results[filtered_results['crossencoder_candidates'].apply(lambda x: len(x[0]) != 0 if x else False)]

# filtered_results = filtered_results.iloc[:500]
filtered_results

,document_id,offsets,deabbreviated_text,db_ids,mention_id,joined_offsets,biencoder_candidates,biencoder_hit_index,crossencoder_candidates,crossencoder_hit_index
0,9288106,"[[40, 61]]",ataxia-telangiectasia,[MESH:D001260],9288106.1,"40,61","[[MESH:D001260, OMIM:208900], [MESH:D057891], ...",0,"[[MESH:D001260, OMIM:208900], [MESH:D057891], ...",0
2,9288106,"[[99, 120]]",Ataxia-telangiectasia,[MESH:D001260],9288106.3,"99,120","[[MESH:D001260, OMIM:208900], [MESH:D057891], ...",0,"[[MESH:D001260, OMIM:208900], [MESH:D057891], ...",0
3,9288106,"[[122, 125]]",Ataxia-telangiectasia,[MESH:D001260],9288106.4,"122,125","[[MESH:D001260, OMIM:208900], [MESH:D019294, O...",0,"[[MESH:D001260, OMIM:208900], [MESH:C566865], ...",0
4,9288106,"[[132, 163]]",recessive multi-system disorder,[MESH:D030342],9288106.5,"132,163","[[MESH:D030342], [MESH:D004194], [MESH:D001523...",0,"[[MESH:D030342], [MESH:D058495, OMIM:117550], ...",0
5,9288106,"[[235, 241]]",cancer,[MESH:D009369],9288106.6,"235,241","[[MESH:D009369], [MESH:D018198], [MESH:D002471...",0,"[[MESH:D009369], [MESH:D002277], [MESH:D009386...",0
...,...,...,...,...,...,...,...,...,...,...
955,9988281,"[[996, 1015]]",breast malignancies,[MESH:D001943],9988281.7,"996,1015","[[MESH:D001943, OMIM:114480], [MESH:D058922], ...",0,"[[MESH:D001943, OMIM:114480], [MESH:D009369], ...",0
956,9988281,"[[1123, 1147]]",invasive lobular cancers,[MESH:D018275],9988281.8,"1123,1147","[[MESH:D018275], [MESH:D018299], [MESH:D009369...",0,"[[MESH:D018275], [MESH:D009369], [MESH:D002277...",0
957,9988281,"[[1152, 1179]]",low-grade ductal carcinomas,[MESH:D044584],9988281.9,"1152,1179","[[MESH:D044584], [MESH:D018270], [MESH:D002277...",0,"[[MESH:D044584], [MESH:D018270], [MESH:D002277...",0
958,9988281,"[[1269, 1286]]",ductal carcinomas,[MESH:D044584],9988281.10,"1269,1286","[[MESH:D044584], [MESH:D018270], [MESH:D002277...",0,"[[MESH:D044584], [MESH:D018270], [MESH:D002292...",0


In [24]:
total_nb_mentions = len(results)
total_nb_mentions

960

In [25]:
# def number_hit(df, hit_index_column, hit_range):
#     ''' 
#     Count the number of hits for each hit index.
#     ------
#     df : DataFrame
#     hit_index_column : str (column name of the hit index)
#     hit_range : int (Number of hits to consider = recall@k)
#     '''
#     results = {i+1: 0 for i in range(hit_range)}
#     res = 0
#     for hit_index in range(hit_range):
#         for idx, row in df.iterrows():
#             if row[hit_index_column] == hit_index:
#                 res += 1
#         results[hit_index+1] = res
#     return results

number_hits_biencoder = number_hit(filtered_results, 'biencoder_hit_index', number_candidates)
number_hits_crossencoder = number_hit(filtered_results, 'crossencoder_hit_index', number_candidates)

In [26]:
# print('number hits biencoder :')
# number_hits_biencoder
print('number hits crossencoder :')
number_hits_crossencoder

number hits crossencoder :


{1: 740,
 2: 767,
 3: 775,
 4: 785,
 5: 787,
 6: 791,
 7: 793,
 8: 796,
 9: 797,
 10: 799,
 11: 800,
 12: 801,
 13: 802,
 14: 803,
 15: 803,
 16: 803,
 17: 803,
 18: 803,
 19: 804,
 20: 804,
 21: 807,
 22: 807,
 23: 808,
 24: 808,
 25: 808,
 26: 808,
 27: 808,
 28: 809,
 29: 809,
 30: 809,
 31: 809,
 32: 809,
 33: 809,
 34: 809,
 35: 809,
 36: 809,
 37: 809,
 38: 809,
 39: 809,
 40: 809,
 41: 809,
 42: 809,
 43: 809,
 44: 809,
 45: 809,
 46: 809,
 47: 809,
 48: 809,
 49: 809,
 50: 809,
 51: 809,
 52: 809,
 53: 809,
 54: 809,
 55: 809,
 56: 809,
 57: 809,
 58: 809,
 59: 809,
 60: 809,
 61: 809,
 62: 809,
 63: 809,
 64: 809}

In [27]:
# def compute_recall(df, hit_index_column, hit_range, total_nb_mentions):
#     ''' 
#     Function to compute the recall of the intial model (biencoder, crossencoder).
#     ------
#     df : DataFrame
#     hit_index_column : str ("biencoder_hit_index" or "crossencoder_hit_index")
#     hit_range : int (Number of hits to consider = recall@k)
#     total_nb_mentions : int (For unnormalized result = true performance)
#     '''
#     unnormalized_recall = 0
#     normalized_recall = 0
#     results = []
    
#     for hit_index in range(hit_range):
#         res = 0
#         for idx, row in df.iterrows():
#             if row[hit_index_column] == hit_index:
#                 res += 1
#         unnormalized_recall += res / total_nb_mentions
#         normalized_recall += res / len(df)

#         # Store the cumulative results for this hit_index
#         results.append((unnormalized_recall, normalized_recall))
    
#     return results

biencoder_results = compute_recall(filtered_results, "biencoder_hit_index", 5, total_nb_mentions)
print("Biencoder:")
for i, (unnormalized, normalized) in enumerate(biencoder_results):
    print(f"recall {i+1}: Normalized = {normalized:.4f}, Unnormalized = {unnormalized:.4f}")

crossencoder_results = compute_recall(filtered_results, "crossencoder_hit_index", 5, total_nb_mentions)
print("Crossencoder:")
for i, (unnormalized, normalized) in enumerate(crossencoder_results):
    print(f"recall {i+1}: Normalized = {normalized:.4f}, Unnormalized = {unnormalized:.4f}")


Biencoder:
recall 1: Normalized = 0.8714, Unnormalized = 0.7344
recall 2: Normalized = 0.9159, Unnormalized = 0.7719
recall 3: Normalized = 0.9333, Unnormalized = 0.7865
recall 4: Normalized = 0.9431, Unnormalized = 0.7948
recall 5: Normalized = 0.9481, Unnormalized = 0.7990
Crossencoder:
recall 1: Normalized = 0.9147, Unnormalized = 0.7708
recall 2: Normalized = 0.9481, Unnormalized = 0.7990
recall 3: Normalized = 0.9580, Unnormalized = 0.8073
recall 4: Normalized = 0.9703, Unnormalized = 0.8177
recall 5: Normalized = 0.9728, Unnormalized = 0.8198


# Part with GPT

### Train set

In [22]:

train_mentions = []
train_mention2context = {}
train_mention2gold = {}
train_mention2text = {}
for idx, row in train_df.iterrows():
    train_mention2gold[row['mention_id']] = row['db_ids']
    train_mentions.append(row['mention_id'])
    train_mention2text[row['mention_id']] = row['deabbreviated_text']
    train_mention2context[row["mention_id"]] = row["limited_contextualized_mention"]

# train_mention2gold

### Test set

In [23]:
mention2context = {}
for idx, row in test_df.iterrows():
    mention2context[row["mention_id"]] = row["limited_contextualized_mention"]

mentions = []
mention2biencoder_candidates = {}
mention2crossencoder_candidates = {}
mention2gold = {}
mention2hit = {}
mention2text = {}
for idx, row in filtered_results.iterrows():
    # print(idx)
    # Only consider row if hit_index < max number of candidates
    if row['biencoder_hit_index'] < number_candidates:
        mention2biencoder_candidates[row['mention_id']] = [el[0] for el in row['biencoder_candidates'][:number_candidates]]
        # mention2crossencoder_candidates[row['mention_id']] = [el[0] for el in row['crossencoder_candidates'][:number_candidates]]
        mention2gold[row['mention_id']] = row['db_ids']
        mentions.append(row['mention_id'])
        mention2text[row['mention_id']] = row['deabbreviated_text']
        mention2hit[row['mention_id']] = row['biencoder_hit_index']
print(len(mention2hit))
mention2hit
# len(test_df)

817


{'9288106.1': 0,
 '9288106.2': 1,
 '9288106.3': 0,
 '9288106.4': 0,
 '9288106.5': 0,
 '9288106.6': 0,
 '9288106.7': 0,
 '9288106.8': 0,
 '9288106.9': 0,
 '9288106.10': 1,
 '9288106.11': 0,
 '9288106.12': 43,
 '9288106.13': 1,
 '9288106.14': 0,
 '9288106.15': 0,
 '9288106.16': 0,
 '9288106.17': 0,
 '9288106.18': 0,
 '9288106.19': 0,
 '9288106.20': 0,
 '9288106.21': 0,
 '9288106.22': 0,
 '9288106.23': 2,
 '9288106.24': 2,
 '9288106.25': 2,
 '9288106.26': 0,
 '9288106.28': 0,
 '9288106.29': 1,
 '9294109.1': 0,
 '9294109.2': 0,
 '9294109.3': 0,
 '9294109.4': 1,
 '9294109.5': 0,
 '9311732.1': 0,
 '9311732.2': 0,
 '9311732.3': 0,
 '9311732.4': 0,
 '9311732.5': 0,
 '9311732.6': 0,
 '9311732.7': 0,
 '9311732.8': 0,
 '9311732.9': 0,
 '9311732.10': 0,
 '9311732.11': 0,
 '9311732.12': 0,
 '9311732.13': 0,
 '9311732.14': 0,
 '9311732.15': 0,
 '932197.3': 0,
 '932197.9': 0,
 '932197.10': 0,
 '932197.11': 0,
 '932197.12': 0,
 '932197.14': 0,
 '9336417.1': 0,
 '9336417.2': 0,
 '9336417.3': 0,
 '93364

In [24]:
# def get_candidates_name(candidates, ontology):
#     ''' 
#     Returns the name of the candidates
#     ------
#     candidates : list of list of CUIs : [[cui1], [cui2, cui3], ...]
#     ontology : BiomedicalOntology object
#     '''
#     candidates_name = {}
#     for candidate in candidates:
#         entity = ontology.entities.get(candidate)
#         candidates_name[entity.cui] = entity.name
#     return candidates_name



# def get_candidates_data(candidates, ontology):
#     ''' 
#     Returns the metadata of the candidates
#     ------
#     candidates : list of list of CUIs : [[cui1], [cui2, cui3], ...]
#     ontology : BiomedicalOntology object
#     '''
#     candidates_data = {}
#     for candidate in candidates:
#         entity = ontology.entities.get(candidate)
#         if entity:
#             entity_data = {
#                 'cui': entity.cui,
#                 'name': entity.name,
#                 'types': entity.types,
#                 'aliases': entity.aliases,
#                 'definition': entity.definition
#             }
#             candidates_data[entity.cui] = entity_data
#         else:
#             candidates_data[candidate] = {'error': f'Entity for {candidate} not found'}
#     return candidates_data

# res = get_candidates_data(candidates=mention2biencoder_candidates[mentions[0]], 
#                           ontology=ontology2)
# res
# len(res)
# # mention2biencoder_candidates[mentions[0]]

# # def get_candidates_data_v2(candidates) :  
# #     candidates_data = []
# #     for candidate in candidates:
# #         entity = candidate + " : " + cui2description[candidate]
# #         candidates_data.append(entity)
# #     return candidates_data

# # get_candidates_data_v2(mention2biencoder_candidates[mentions[0]])

In [25]:
# '''
# Create a string for gpt prompt with : 
# "Query : ... / Sentence (context+mention) : ... / Answer : cui
# etc...
# Query : ... / Sentence (context+mention) : ... / Answer : cui"
# '''
# def topk_examples(model,
#                   index,
#                   query, 
#                   corpus, 
#                   TrainMap_context2mention, 
#                   train_mention2text, 
#                   train_mention2gold, 
#                   ontology,
#                   k = 5) : 
#     ''' 
#     Given a query (context sentence), returns the top k most similar contexts from the corpus.
#     ------
#     query : str (context sentence)
#     corpus : list of str (all context sentences)
#     TrainMap_context2mention : dict (context sentence to mention_id)
#     train_mention2text : dict (mention_id to mention name)
#     ontology : BiomedicalOntology object
#     k : int (number of nearest neighbors)
#     '''
#     indices = knn_query(model, index, query, k)
#     result_list = []
#     for i, idx in enumerate(indices) :
#         NN_mention = corpus[idx]
#         # print("Nearest neighbor mention : ", NN_mention)
#         if NN_mention not in TrainMap_context2mention :
#             continue
#         mention_id = TrainMap_context2mention[NN_mention]
#         # print("mention_id :", mention_id)
#         mention_text = train_mention2text[mention_id]
#         cui = train_mention2gold[mention_id]
#         # print("cui :", cui)
#         gold = get_candidates_data(candidates=cui, ontology=ontology)
#         # print("gold :", gold)
#         result_list.append(f"Mention {i+1}: {mention_text} || Context: {NN_mention} || Correct CUI: {gold}")
    
#     res = "\n".join(result_list)
    
#     return res

In [26]:
system_instructions = """You are a professional data annotator and curator.
Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of {number_candidates} candidate entities."""

system_instructions_recall = """You are a professional data annotator and curator.
Your task is to rank the candidate entities from best to worst for a given mention based on the provided context and the descriptions of each candidate entities."""


In [27]:
# def prompt_gpt(mention, 
#             context, 
#             candidates, 
#             system_instructions, 
#             topk_examples, 
#             llm = "gpt-4o-mini"): 
#     ''' 
#     mention : str (name of the mention to be linked)
#     context : str (context where the mention appears)
#     candidates : list of list of CUIs : [[cui1], [cui2, cui3], ...]
#     system_instructions : str (instructions for the LLM)
#     topk_examples : str (top k examples of similar contexts)
#     model : str (name of the model to use)
#     '''
#     completion = openai.chat.completions.create(
#         model = llm,
#         messages=[
#             {"role": "system", "content": system_instructions},
#             {
#                 "role": "user",
#                 "content": f""" 
#                 Here are a few examples : \n
#                 {topk_examples} \n

#                 This is the specific mention that needs to be linked to the correct entity : {mention} \n
                
#                 This is the context where the mention appears : {context} \n
                
#                 These are the candidate entities to choose from: {candidates} \n
#                 You must provide an answer among the candidates. \n
                
#                 Return the answer in the following format : CUI \n
                
#                 Do not add any explanations !
#                 """
#             },
#         ],
#         max_tokens= 2048,
#         temperature= 0,
#     )
#                 # Do not add any explanations ! 
#                 # Do not return anything except the correct CUI from the list of candidates.
#                 # If there are multiple possible entities, you can return multiple IDs.
                
#                 # Be careful not to mix it with another mention that could appear in the context !
                
#                 # This is the specific mention that needs to be linked to the correct entity : {mentions[i]}
#     return completion.choices[0].message.content


In [35]:
# def evaluate_gpt(mentions, 
#              mention2context, 
#              mention2biencoder_candidates, 
#              ontology, 
#              k, 
#              corpus,
#              TrainMap_context2mention,
#              train_mention2text,
#              train_mention2gold,
#              system_instructions,
#              index,
#              nlp_model,
#              llm = "gpt-4o-mini"):
#     ''' 
#     Run "prompt" function for each mention in the list of mentions.
#     Returns a dictionary {mention_id : predicted CUI}
#     -------
#     mentions : list (mention_ids)
#     ontology : BiomedicalOntology object
#     mention2context : dict (mention_id : context)
#     mention2biencoder_candidates : dict (mention_id : list of candidate CUIs)
#     k : int (number of nearest neighbors)
#     corpus : list of str (all context sentences)
#     TrainMap_context2mention : dict (context sentence to mention_id)
#     train_mention2text : dict (mention_id to mention name)
#     train_mention2gold : dict (mention_id to gold CUI)
#     system_instructions : str (instructions for the LLM)
#     index : faiss index
#     nlp_model : sentence-transformers model
#     llm : str (name of the model)
#     '''
#     results = {}
#     for i in range(len(mentions)) :
#         mention = mentions[i]
#         context = mention2context[mention]
#         candidates = get_candidates_data(candidates=mention2biencoder_candidates[mentions[i]], 
#                                          ontology=ontology)
#         # candidates = get_candidates_data_v2(mention2crossencoder_candidates[mention])
#         topk = topk_examples(model = nlp_model,
#                             index = index,
#                             query=context, 
#                             corpus=corpus, 
#                             TrainMap_context2mention=TrainMap_context2mention, 
#                             train_mention2text=train_mention2text, 
#                             train_mention2gold=train_mention2gold, 
#                             ontology=ontology, 
#                             k=k)

#         text = prompt_gpt(mention = mention, 
#                           context = context, 
#                           system_instructions=system_instructions,
#                           candidates = candidates, 
#                           topk_examples=topk, 
#                           llm = llm)
        
#         cand = extract_cui(text)
#         results[mention] = cand
        
#         if i%20 == 0:
#             print(f"i = {i}")

#     return results
    


In [36]:
# def scoring(results, mention2gold) :
#     ''' 
#     Return the score of of the model
#     -------
#     results : dictionary {mention_id : predicted CUI}
#     mention2gold : dictionary {mention_id : gold CUI}
#     '''
#     score = 0
#     for key, value in results.items():
#         if value in mention2gold[key] :
#             score += 1

#     return score/len(results)

In [51]:
# def error_analysis(results, ontology, mention2gold, mention2context) :
#     ''' 
#     Returns a dict of mention_id for mentions that were not correctly predicted
#     Each dict contains the gold cui, the predicted cui, the context of the mention.
#     -------
#     results : dictionary {mention_id : predicted CUI}
#     ontology : BiomedicalOntology object
#     mention2gold : dictionary {mention_id : gold CUI}
#     mention2context : dictionary {mention_id : context}
#     '''
#     error_mentions = defaultdict(dict)
#     for mention_id, predicted_cui in results.items() :
#         gold_cui = mention2gold[mention_id]
#         if predicted_cui not in gold_cui :
#             predicted_cui_metadata = get_candidates_data(candidates=[predicted_cui], 
#                                                          ontology=ontology)
#             gold_cui_metadata = get_candidates_data(candidates=[gold_cui[0]], 
#                                                     ontology=ontology)
#             mention_context = mention2context[mention_id]
#             error_mentions[mention_id] = {"query_context": mention_context, "gold_cui": gold_cui_metadata, "predicted_cui": predicted_cui_metadata}

#     return error_mentions

### Single prompt for error analysis

In [ ]:
# mention = '9931324.9' # "9973276.8"
mention = "9288106.17"
# i = 39
# mention = mentions[i]
text = mention2text[mention]
context = mention2context[mention]
candidates = get_candidates_data(candidates = mention2biencoder_candidates[mention], 
                                 ontology = ontology2)
topk = topk_examples(model = model,
                    index = index,
                    query=context, 
                    corpus=corpus, 
                    TrainMap_context2mention=TrainMap_context2mention, 
                    train_mention2text=train_mention2text, 
                    train_mention2gold=train_mention2gold, 
                    ontology=ontology2, 
                    k=3)
# random.shuffle(candidates)
# candidates = get_candidates_data_v2(mention2candidates[mentions[i]])

pred_cui = prompt_gpt(mention = text, 
                      context=context, 
                      candidates=candidates,
                      system_instructions=system_instructions, 
                      topk_examples=topk, 
                      llm = "gpt-4o-mini")
# model = "gpt-3.5-turbo-0125"
# model = "gpt-4o-2024-08-06"
print("Mention ID :", mention)
print("Context :", context)
print("Top k examples :", topk)
print("Text :", text)
print("Gold CUI :", mention2gold[mention])

print("Predicted CUI :", pred_cui)

In [ ]:
mention = "9288106.17"
# i = 39
# mention = mentions[i]
text = mention2text[mention]
context = mention2context[mention]
candidates = get_candidates_data(candidates = mention2biencoder_candidates[mention], 
                                 ontology = ontology2)
topk = topk_examples(model = model,
                    index = index,
                    query=context, 
                    corpus=corpus, 
                    TrainMap_context2mention=TrainMap_context2mention, 
                    train_mention2text=train_mention2text, 
                    train_mention2gold=train_mention2gold, 
                    ontology=ontology2, 
                    k=3)
print(f"""
You are a professional data annotator and curator.
Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of 20 candidate entities.
      
Here are a few examples : \n
{topk} \n

This is the specific mention that needs to be linked to the correct entity : {text} \n

This is the context where the mention appears : {context} \n

These are the candidate entities to choose from: {candidates} \n
You must provide an answer among the candidates. \n

Return the answer in the following format : CUI \n
For instance : "MESH:D000000" "OMIM:000000" are valid answers. \n
Reason step by step but do not add provide any explanations to me ! I only want the final answer.
""")

In [ ]:
print("context :", context)
print("topk :", topk)

In [32]:
results = evaluate_gpt(llm = "gpt-4o-mini",
                    nlp_model=model,
                    index=index,
                    mentions=mentions[:5], 
                    mention2context=mention2context,
                    mention2biencoder_candidates=mention2biencoder_candidates,   
                    ontology=ontology2, 
                    k = 3, 
                    corpus=corpus,
                    TrainMap_context2mention=TrainMap_context2mention,
                    train_mention2text=train_mention2text,
                    train_mention2gold=train_mention2gold,
                    system_instructions=system_instructions)
results

Batches: 100%|██████████| 1/1 [00:00<00:00, 129.52it/s]

(1, 768)


i = 0


Batches: 100%|██████████| 1/1 [00:00<00:00, 92.59it/s]

(1, 768)


...

Batches: 100%|██████████| 1/1 [00:00<00:00, 68.30it/s]


(1, 768)


Batches: 100%|██████████| 1/1 [00:00<00:00, 127.94it/s]


(1, 768)


{'9288106.1': 'MESH:D001260',
 '9288106.2': 'MESH:D015458',
 '9288106.3': 'MESH:D001260',
 '9288106.4': 'MESH:D001260',
 '9288106.5': 'MESH:D030342'}

In [33]:
score = scoring(results=results, mention2gold=mention2gold)
score


1.0

In [31]:
### SAVE THE RESULTS ! IT'S EXPENSIVE TO RUN THE MODEL !!!

# with open("gpt4o_results.json", "w") as f:
#     json.dump(results, f, indent=4)

with open("gpt4o_results.json", "r") as f:
    results = json.load(f)

# results

In [35]:
error_mentions = error_analysis(results = results, 
                                ontology = ontology2, 
                                mention2gold = mention2gold, 
                                mention2context = mention2context)
error_mentions

defaultdict(dict, {})

# Evaluation with Mixtral

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

mixtral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")
mixtral_model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")


### ICL

In [44]:
def prompt_mistral(mention, context, candidates, topk_examples):
    ''' 
    Equivalent of the prompt function for the Mistral model.
    '''
    
    prompt_text = f"""
    You are a professional data annotator and curator.
    Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of {number_candidates} candidate entities. \n
    
    Here are a few examples : \n
    {topk_examples} \n

    This is the specific mention that needs to be linked to the correct entity : {mention} \n
    
    This is the context where the mention appears : \n
    {context} \n
    
    These are the candidate entities to choose from: \n
    {candidates} \n
    
    You must provide an answer among the candidates. \n
    
    Return the answer in the following format : CUI
    Do not add any explanations!
    """

    # Tokenize input
    inputs = mixtral_tokenizer(prompt_text, return_tensors="pt", max_length=2048, truncation=True)

    # Generate output (you can adjust `max_new_tokens` to limit the output)
    with torch.no_grad():
        outputs = mixtral_model.generate(**inputs, max_new_tokens=300, temperature=0.0)

    # Decode the generated tokens
    generated_text = mixtral_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the CUI answer (since the model generates the entire text, you might need to clean it)
    return generated_text.strip()

In [45]:
# mention = '9931324.9' # "9973276.8"
mention = "9888388.6"
# i = 39
# mention = mentions[i]
text = mention2text[mention]
context = mention2context[mention]
candidates = get_candidates_data(candidates=mention2biencoder_candidates[mention], 
                                 ontology=ontology2)
topk = topk_examples(query=context, 
                        corpus=corpus, 
                        TrainMap_context2mention=TrainMap_context2mention, 
                        train_mention2text=train_mention2text, 
                        train_mention2gold=train_mention2gold, 
                        ontology=ontology2, 
                        k=3)

pred_cui = prompt_mistral(text, context, candidates, topk_examples=topk)

print("Predicted CUI :", pred_cui)

Batches: 100%|██████████| 1/1 [00:00<00:00, 92.61it/s]
/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


(1, 768)
Predicted CUI : You are a professional data annotator and curator.
    Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of 64 candidate entities. 

    
    Here are a few examples : 

    Mention 1: peroxisomal disease || Context: Determination of 30 X-linked adrenoleukodystrophy mutations, including 15 not previously described. X-linked Adrenoleukodystrophy (X-ALD) is the most frequent [ENTITY_START] peroxisomal disease. [ENTITY_END] It mainly involves the nervous system white matter, adrenal cortex and testes. Several distinct clinical phenotypes are known. The principal biochemical abnormality is the accumulation of saturated very-long-chain fatty acids (VLCFAs || Correct CUI: {'MESH:D018901': {'cui': 'MESH:D018901', 'name': 'Peroxisomal Disorders', 'types': 'Disease', 'aliases': 'Acidemia, Hyperpipecolic|Acidemias, Hyperpipecolic|Adrenoleukodystrophies, Neonatal|Adrenoleukodystrophy, Autosomal Neonatal Form

# Flan T5 model

In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load Flan-T5 model and tokenizer
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-xxl")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-xxl")


/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  1.74it/s]


In [29]:
def prompt_flan_t5(mention, context, candidates, topk_examples):
    prompt_text = f"""
    You are a professional data annotator and curator.
    Your task is to identify the correct entity for a given mention based on the provided context and the descriptions of 64 candidate entities.

    Here are a few examples: \n
    {topk_examples} \n

    This is the specific mention that needs to be linked to the correct entity: {mention} \n

    This is the context where the mention appears : \n
    {context} \n
    
    These are the candidate entities to choose from: \n
    {candidates} \n
    
    You must provide an answer among the candidates. \n

    Return the answer in the following format: CUI
    Do not add any explanations!
    """

    inputs = flan_tokenizer(prompt_text, return_tensors="pt", max_length=2048, truncation=True)

    outputs = flan_model.generate(**inputs, max_new_tokens=100, temperature=0.0)
    generated_text = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text.strip()


In [ ]:
mention = "9988281.10"
text = mention2text[mention]
context = mention2context[mention]
candidates = get_candidates_data(mention2biencoder_candidates[mention], ontology2)
topk = topk_examples(query=context, 
                        corpus=corpus, 
                        TrainMap_context2mention=TrainMap_context2mention, 
                        train_mention2text=train_mention2text, 
                        train_mention2gold=train_mention2gold, 
                        ontology=ontology2, 
                        k=3)

pred_cui = prompt_flan_t5(mention = text, context = context, candidates=candidates, topk_examples=topk)

print("Mention ID :", mention)
print("Context :", context)
print("Top k examples :", topk)
print("Text :", mention2text[mention])
print("Gold CUI :", mention2gold[mention])
print("Predicted CUI :", pred_cui)

In [84]:
CUIs = [v[0] for k, v in mention2gold.items()] # List of all gold CUIs from the test set (reduced one = only with hit_index < nb_candidates)

cui2name = get_candidates_name(CUIs, ontology2)
name2cui = {v: k for k, v in cui2name.items()}

name2cui[pred_cui]

'MESH:D009422'

# vllm
# LLAMA
Llama-3.1-8B-UltraMedical (71%)

Bio-Medical-Llama-3-8B (41%)

Laim/Llama-3.1-MedPalm2-imitate-8B-Instruct (17%)

meta-llama/Meta-Llama-3.1-8B-Instruct (79%) (91% for k=10)

mistralai/Mistral-7B-Instruct-v0.3 (82%) (84% for k=10)

mistralai/Mistral-Nemo-Instruct-2407 (90%)

ISTA-DASLab/Mixtral-8x7B-Instruct-v0_1-AQLM-2Bit-1x16-hf

In [26]:
# llm = LLM(
#     model="meta-llama/Meta-Llama-3.1-8B-Instruct",
#     enforce_eager=True,
# )

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
device_1 = torch.device("cuda:0")  # First GPU (GPU 0)
device_2 = torch.device("cuda:1")  # Second GPU (GPU 1)

llm = LLM(
model="Qwen/Qwen2.5-7B-Instruct",
tensor_parallel_size=1,
dtype = "half",
gpu_memory_utilization=0.85,
max_logprobs=1000,
device=device_1,
max_model_len = 30000)

tokenizer = llm.get_tokenizer()

WARNING 09-21 19:00:15 config.py:1651] Casting torch.bfloat16 to torch.float16.
WARNING 09-21 19:00:15 config.py:365] Async output processing is only supported for CUDA or TPU. Disabling it for other platforms.
INFO 09-21 19:00:15 llm_engine.py:213] Initializing an LLM engine (v0.6.0) with config: model='Qwen/Qwen2.5-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=30000, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda:0, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collec

/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 09-21 19:00:16 model_runner.py:915] Starting to load model Qwen/Qwen2.5-7B-Instruct...
INFO 09-21 19:00:16 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 09-21 19:00:16 selector.py:116] Using XFormers backend.
INFO 09-21 19:00:16 weight_utils.py:236] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:04,  1.43s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:02<00:02,  1.42s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:04<00:01,  1.47s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:05<00:00,  1.50s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:05<00:00,  1.48s/it]



INFO 09-21 19:00:23 model_runner.py:926] Loading model weights took 14.2487 GB
INFO 09-21 19:00:29 gpu_executor.py:122] # GPU blocks: 22472, # CPU blocks: 4681
INFO 09-21 19:00:33 model_runner.py:1217] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 09-21 19:00:33 model_runner.py:1221] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 09-21 19:00:59 model_runner.py:1335] Graph capturing finished in 26 secs.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [27]:
# llm2 = LLM(
# model="mistralai/Mistral-7B-Instruct-v0.3",
# tensor_parallel_size=1,
# dtype = "half",
# gpu_memory_utilization=0.9,
# max_logprobs=1000,
# device=device_2)

In [28]:
# torch.cuda.set_device(0)
# torch.cuda.empty_cache()

In [29]:

# def prompt_vllm(mention, 
#                 context, 
#                 system_instructions, 
#                 candidates, 
#                 topk_examples, 
#                 llm, 
#                 tokenizer, 
#                 sampling_params,
#                 reasoning=False):
#     ''' 
#     mention : str (name of the mention to be linked)
#     context : str (context where the mention appears)
#     system_instructions : str (instructions for the LLM)
#     candidates : list of list of CUIs : [[cui1], [cui2, cui3], ...]
#     topk_examples : str (top k examples of similar contexts)
#     llm : LLM model
#     tokenizer : AutoTokenizer
#     sampling_params : SamplingParams config
#     reasoning : bool (whether to use reasoning or not)
#     '''
#     prompt_text = f"""
#     Here are a few examples: \n
#     {topk_examples} \n

#     This is the specific mention that needs to be linked to the correct entity: {mention} \n

#     This is the context where the mention appears: \n
#     {context} \n
    
#     These are the candidate entities to choose from: \n
#     {candidates} \n
    
#     You MUST PROVIDE an ANSWER among the candidates. \n

#     Return the answer in the following format: CUI
#     For instance : "MESH:D000000" "OMIM:000000" are valid answers. \n
#     Reason step by step but do not add provide any explanations to me ! I only want the final answer.
#     """
#     # Do not add provide any explanations ! But you must give AN answer.

#     messages = [
#     {"role": "system", "content": system_instructions},
#     {"role": "user", "content": prompt_text},
#     ]
#     prompts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

#     # Decode the generated tokens into text
#     outputs = llm.generate(prompts=prompts, sampling_params=sampling_params)
#     answer = outputs[0].outputs[0].text

#     return answer


In [31]:
mention_id = "9448273.1"
mention_name = mention2text[mention_id]
context = mention2context[mention_id]
candidates = get_candidates_data(mention2biencoder_candidates[mention_id], ontology2)
topk = topk_examples(model = model, # sentence transformer model
                    index=index,
                    query=context,
                    corpus=corpus,
                    TrainMap_context2mention=TrainMap_context2mention,
                    train_mention2text=train_mention2text,
                    train_mention2gold=train_mention2gold,
                    ontology=ontology2,
                    k=3)

prompt = generate_prompt_text(mention = mention_name,
                              context=context,
                              candidates=candidates,
                              topk_examples=topk,
                              recall=False,
                              recall_k=5,
                              mention_id=mention_id)

# Generate the prediction using LLaMA 8B
pred_cui = prompt_vllm(prompt = prompt,
                       system_instructions=system_instructions,
                       llm=llm,
                       tokenizer=tokenizer,
                       sampling_params=sampling_params)
answer = extract_last_cui(pred_cui)

# Output the results
print("Mention ID:", mention_id)
print("Context:", context)
print("Top k examples:", topk)
print("Text:", mention2text[mention_id])
print("Gold CUI:", mention2gold[mention_id])
print("Predicted CUI:", pred_cui)
print("Final answer :", answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 113.81it/s]


(1, 768)


Processed prompts: 100%|██████████| 1/1 [00:05<00:00,  5.45s/it, est. speed input: 3876.33 toks/s, output: 1.83 toks/s]

Mention ID: 9448273.1
Context: The [ENTITY_START] von Hippel-Lindau tumor [ENTITY_END] suppressor gene is required for cell cycle exit upon serum withdrawal. The inactivation of the von Hippel-Lindau (VHL) tumor suppressor gene predisposes affected individuals to the human VHL cancer syndrome
Top k examples: Mention 1: von Hippel-Lindau || Context: Constitutional [ENTITY_START] von Hippel-Lindau [ENTITY_END] (VHL) gene deletions detected in VHL families by fluorescence in situ hybridization. von Hippel-Lindau (VHL) disease is an autosomal dominantly inherited cancer syndrome predisposing to a variety of tumor types that || Correct CUI: {'MESH:D006623': {'cui': 'MESH:D006623', 'name': 'von Hippel-Lindau Disease', 'types': 'Disease', 'aliases': "Angiomatoses, Familial Cerebelloretinal|Angiomatoses, Familial Cerebello-Retinal|Angiomatosis, Familial Cerebelloretinal|Angiomatosis, Familial Cerebello-Retinal|Angiomatosis Retinae|Cerebelloretinal Angiomatoses, Familial|Cerebello-Retinal Angio

In [44]:
print(prompt)


        Here are a few examples:
Mention 1: von Hippel-Lindau || Context: Constitutional [ENTITY_START] von Hippel-Lindau [ENTITY_END] (VHL) gene deletions detected in VHL families by fluorescence in situ hybridization. von Hippel-Lindau (VHL) disease is an autosomal dominantly inherited cancer syndrome predisposing to a variety of tumor types that || Correct CUI: {'MESH:D006623': {'cui': 'MESH:D006623', 'name': 'von Hippel-Lindau Disease', 'types': 'Disease', 'aliases': "Angiomatoses, Familial Cerebelloretinal|Angiomatoses, Familial Cerebello-Retinal|Angiomatosis, Familial Cerebelloretinal|Angiomatosis, Familial Cerebello-Retinal|Angiomatosis Retinae|Cerebelloretinal Angiomatoses, Familial|Cerebello-Retinal Angiomatoses, Familial|Cerebelloretinal Angiomatosis, Familial|Cerebello-Retinal Angiomatosis, Familial|Familial Cerebelloretinal Angiomatoses|Familial Cerebello-Retinal Angiomatoses|Familial Cerebelloretinal Angiomatosis|Familial Cerebello Retinal Angiomatosis|Familial Cerebello-

In [36]:
# def evaluate_vllm(
#     llm,
#     nlp_model,
#     tokenizer,
#     index,
#     system_instructions,
#     mentions,
#     ontology,
#     corpus,
#     mention2context,
#     mention2biencoder_candidates,
#     mention2text,
#     TrainMap_context2mention,
#     train_mention2text,
#     train_mention2gold,
#     k,
#     sampling_params,
#     reasoning = False,
# ):
#     """
#     Run "prompt" function for each mention in the list of mentions.
#     Returns a dictionary {mention_id : predicted CUI}
#     -------
#     llm : LLM model
#     nlp_model : SentenceTransformer model
#     tokenizer : AutoTokenizer
#     index : faiss index
#     system_instructions : str (instructions for the LLM)
#     mentions : list (mention_ids)
#     ontology : BiomedicalOntology object
#     corpus : list of str (all context sentences)
#     mention2context : dict (mention_id : context)
#     mention2biencoder_candidates : dict (mention_id : list of candidate CUIs)
#     mention2text : dict (mention_id : mention name)
#     TrainMap_context2mention : dict (context sentence to mention_id)
#     train_mention2text : dict (mention_id to mention name)
#     train_mention2gold : dict (mention_id to gold CUI)
#     k : int (number of nearest neighbors)
#     sampling_params : SamplingParams config
#     reasoning : bool (whether to use reasoning or not)
#     """
#     results = {}
#     for i in range(len(mentions)):
#         mention_id = mentions[i]
#         mention_name = mention2text[mention_id]
#         context = mention2context[mention_id]
#         candidates = get_candidates_data(
#             mention2biencoder_candidates[mentions[i]], ontology
#         )
#         # candidates = get_candidates_data_v2(mention2crossencoder_candidates[mention])
#         topk = topk_examples(
#             model=nlp_model,  # sentence transformer model
#             index=index,
#             query=context,
#             corpus=corpus,
#             TrainMap_context2mention=TrainMap_context2mention,
#             train_mention2text=train_mention2text,
#             train_mention2gold=train_mention2gold,
#             ontology=ontology,
#             k=k,
#         )

#         text = prompt_vllm(
#             mention=mention_name,
#             context=context,
#             system_instructions=system_instructions,
#             candidates=candidates,
#             topk_examples=topk,
#             llm=llm,
#             tokenizer=tokenizer,
#             sampling_params=sampling_params,
#             reasoning = reasoning,
#         )
#         print("mention ID :", mention_id, "|| LLM answer :", text)
#         cand = extract_cui(text)
#         results[mention_id] = cand
#         if i % 20 == 0:
#             print(f"i = {i}")

#     return results

In [ ]:
results = evaluate_vllm(llm=llm,
                        nlp_model=model,
                        tokenizer=tokenizer,
                        index=index,
                        system_instructions=system_instructions,
                        mentions=mentions,
                        ontology=ontology2,
                        corpus=corpus,
                        mention2context=mention2context,
                        mention2biencoder_candidates=mention2biencoder_candidates,
                        mention2text=mention2text,
                        TrainMap_context2mention=TrainMap_context2mention,
                        train_mention2text=train_mention2text,
                        train_mention2gold=train_mention2gold,
                        k=3,
                        sampling_params=sampling_params)
results

In [45]:
score = scoring(results=results, 
                mention2gold=mention2gold)
score

1.0

## Result : With reasoning

In [28]:
### SAVE THE RESULTS (It's long to run the model 26mins)
    
# with open("Meta-Llama-3.1-8B-Instruct_k=10_reasoning_results.json", "w") as f:
#     json.dump(results, f, indent=4)

with open("data/biencoder/reasoning2/Meta-Llama-3.1-8B-Instruct_k=3_reasoning_results.json", "r") as f:
    results1r = json.load(f)
    
with open("data/biencoder/reasoning2/Meta-Llama-3.1-8B-Instruct_k=5_reasoning_results.json", "r") as f:
    results2r = json.load(f)
    
with open("data/biencoder/reasoning2/Meta-Llama-3.1-8B-Instruct_k=10_reasoning_results.json", "r") as f:
    results3r = json.load(f)
    
    
###############################################################  
with open("data/biencoder/reasoning2/Mistral-7B-Instruct-v0.3_k=3_reasoning_results.json", "r") as f:
    results4r = json.load(f)

with open("data/biencoder/reasoning2/Mistral-7B-Instruct-v0.3_k=5_reasoning_results.json", "r") as f:
    results5r = json.load(f)
    
with open("data/biencoder/reasoning2/Mistral-7B-Instruct-v0.3_k=10_reasoning_results.json", "r") as f:
    results6r = json.load(f)


###############################################################
with open("data/biencoder/reasoning2/Mistral-Nemo-Instruct-2407_k=3_reasoning_results.json", "r") as f:
    results7r = json.load(f)

with open("data/biencoder/reasoning2/Mistral-Nemo-Instruct-2407_k=5_reasoning_results.json", "r") as f:
    results8r = json.load(f)

with open("data/biencoder/reasoning2/Mistral-Nemo-Instruct-2407_k=10_reasoning_results.json", "r") as f:
    results9r = json.load(f)
    
    
###############################################################
with open("data/biencoder/reasoning2/Qwen2.5-7B-Instruct_k=3_reasoning_results.json", "r") as f:
    results10r = json.load(f)

# with open("data/biencoder/reasoning2/Qwen2.5-7B-Instruct_k=5_reasoning_results.json", "r") as f:
#     results11r = json.load(f)

# with open("data/biencoder/reasoning2/Qwen2.5-7B-Instruct_k=10_reasoning_results.json", "r") as f:
#     results12r = json.load(f)
    
    
###############################################################
with open("data/biencoder/reasoning2/Qwen2.5-14B-Instruct_k=3_reasoning_results.json", "r") as f:
    results13r = json.load(f)

# with open("data/biencoder/reasoning2/Qwen2.5-14B-Instruct_k=5_reasoning_results.json", "r") as f:
#     results14r = json.load(f)

# with open("data/biencoder/reasoning2/Qwen2.5-14B-Instruct_k=10_reasoning_results.json", "r") as f:
#     results15r = json.load(f)

In [29]:
# for mention_id in results.copy() :
#     if results[mention_id] is None:
#         del results[mention_id]

score1r = scoring(results = results1r, mention2gold = mention2gold)
print("Meta-Llama-3.1-8B-Instruct_k=3_reasoning_results :", score1r)
score2r = scoring(results = results2r, mention2gold = mention2gold)
print("Meta-Llama-3.1-8B-Instruct_k=5_reasoning_results:", score2r)
score3r = scoring(results = results3r, mention2gold = mention2gold)
print("Meta-Llama-3.1-8B-Instruct_k=10_reasoning_results:", score3r)

print("---------------------------------")

score4r = scoring(results = results4r, mention2gold = mention2gold)
print("Mistral-7B-Instruct-v0.3_k=3_reasoning_results :", score4r)
score5r = scoring(results = results5r, mention2gold = mention2gold)
print("Mistral-7B-Instruct-v0.3_k=5_reasoning_results :", score5r)
score6r = scoring(results = results6r, mention2gold = mention2gold)
print("Mistral-7B-Instruct-v0.3_k=10_reasoning_results :", score6r)

print("---------------------------------")

score7r = scoring(results = results7r, mention2gold = mention2gold)
print("Mistral-Nemo-Instruct-2407_k=3_reasoning_results :", score7r)
score8r = scoring(results = results8r, mention2gold = mention2gold)
print("Mistral-Nemo-Instruct-2407_k=5_reasoning_results:", score8r)
score9r = scoring(results = results9r, mention2gold = mention2gold)
print("Mistral-Nemo-Instruct-2407_k=10_reasoning_results:", score9r)

print("---------------------------------")

score10r = scoring(results = results10r, mention2gold = mention2gold)
print("Qwen2.5-7B-Instruct_k=3_results :", score10r)
# score11r = scoring(results = results11r, mention2gold = mention2gold)
# print("Qwen2.5-7B-Instruct_k=5_results:", score11r)
# score12r = scoring(results = results12r, mention2gold = mention2gold)
# print("Qwen2.5-7B-Instruct_k=10_results:", score12r)

print("---------------------------------")

score13r = scoring(results = results13r, mention2gold = mention2gold)
print("Qwen2.5-14B-Instruct_k=3_results :", score13r)
# score14r = scoring(results = results14r, mention2gold = mention2gold)
# print("Qwen2.5-14B-Instruct_k=5_results:", score14r)
# score15r = scoring(results = results15r, mention2gold = mention2gold)
# print("Qwen2.5-14B-Instruct_k=10_results:", score15r)

Meta-Llama-3.1-8B-Instruct_k=3_reasoning_results : 0.87875
Meta-Llama-3.1-8B-Instruct_k=5_reasoning_results: 0.87375
Meta-Llama-3.1-8B-Instruct_k=10_reasoning_results: 0.85125
---------------------------------
Mistral-7B-Instruct-v0.3_k=3_reasoning_results : 0.85375
Mistral-7B-Instruct-v0.3_k=5_reasoning_results : 0.8675
Mistral-7B-Instruct-v0.3_k=10_reasoning_results : 0.87125
---------------------------------
Mistral-Nemo-Instruct-2407_k=3_reasoning_results : 0.795
Mistral-Nemo-Instruct-2407_k=5_reasoning_results: 0.81125
Mistral-Nemo-Instruct-2407_k=10_reasoning_results: 0.83625
---------------------------------
Qwen2.5-7B-Instruct_k=3_results : 0.8975
---------------------------------
Qwen2.5-14B-Instruct_k=3_results : 0.88375


In [30]:
error_mentions = error_analysis(results = results13r, 
                                ontology = ontology2, 
                                mention2gold = mention2gold, 
                                mention2context = mention2context)
error_mentions

defaultdict(dict,
            {'9288106.5': {'query_context': 'Clustering of missense mutations in the ataxia-telangiectasia gene in a sporadic T-cell leukaemia. Ataxia-telangiectasia (A-T) is a [ENTITY_START] recessive multi-system disorder [ENTITY_END] caused by mutations in the ATM gene at 11q22-q23 (ref. 3). The risk of cancer, especially lymphoid neoplasias, is substantially elevated in A-T patients and has long been associated with',
              'gold_cui': {'MESH:D030342': {'cui': 'MESH:D030342',
                'name': 'Genetic Diseases, Inborn',
                'types': 'Disease',
                'aliases': 'Defect, Single-Gene|Defects, Single-Gene|Disease, Genetic|Disease, Hereditary|Disease, Inborn Genetic|Diseases, Genetic|Diseases, Hereditary|Diseases, Inborn Genetic|Disorder, Genetic|Disorders, Genetic|Genetic Disease|Genetic Disease, Inborn|Genetic Diseases|Genetic Disorder|Genetic Disorders|Hereditary Disease|Hereditary Diseases|Inborn Genetic Disease|Inborn Genetic D

In [31]:
mention_id = '9988281.10'
print(results13r[mention_id]["explanation"])

To determine the correct entity for the mention "ductal carcinomas," let's analyze the context and the candidate entities provided.

**Context Analysis:**
The context mentions "low-grade ductal carcinomas" and "high-grade ductal carcinomas." This suggests that the term "ductal carcinomas" is being used in the context of breast cancer, specifically referring to a type of breast cancer that originates in the ducts of the breast.

**Candidate Entities Analysis:**
1. **MESH:D044584 - Carcinoma, Ductal**
   - Definition: "Malignant neoplasms involving the ductal systems of any of a number of organs, such as the MAMMARY GLANDS, the PANCREAS, the PROSTATE, or the LACRIMAL GLAND."
   - This is too broad as it includes ductal carcinomas of other organs besides the breast.

2. **MESH:D018270 - Carcinoma, Ductal, Breast**
   - Definition: "An invasive (infiltrating) CARCINOMA of the mammary ductal system (MAMMARY GLANDS) in the human BREAST."
   - This definition aligns well with the context, spe

## Result : Without reasoning

In [32]:
with open("data/biencoder/default2/Meta-Llama-3.1-8B-Instruct_k=3_results.json", "r") as f:
    results1 = json.load(f)
    
with open("data/biencoder/default2/Meta-Llama-3.1-8B-Instruct_k=5_results.json", "r") as f:
    results2 = json.load(f)
    
with open("data/biencoder/default2/Meta-Llama-3.1-8B-Instruct_k=10_results.json", "r") as f:
    results3 = json.load(f)
    
    
###############################################################  
with open("data/biencoder/default2/Mistral-7B-Instruct-v0.3_k=3_results.json", "r") as f:
    results4 = json.load(f)

with open("data/biencoder/default2/Mistral-7B-Instruct-v0.3_k=5_results.json", "r") as f:
    results5 = json.load(f)
    
with open("data/biencoder/default2/Mistral-7B-Instruct-v0.3_k=10_results.json", "r") as f:
    results6 = json.load(f)


###############################################################
with open("data/biencoder/default2/Mistral-Nemo-Instruct-2407_k=3_results.json", "r") as f:
    results7 = json.load(f)

with open("data/biencoder/default2/Mistral-Nemo-Instruct-2407_k=5_results.json", "r") as f:
    results8 = json.load(f)

with open("data/biencoder/default2/Mistral-Nemo-Instruct-2407_k=10_results.json", "r") as f:
    results9 = json.load(f)
    
###############################################################
with open("data/biencoder/default2/Qwen2.5-7B-Instruct_k=3_results.json", "r") as f:
    results10 = json.load(f)

with open("data/biencoder/default2/Qwen2.5-7B-Instruct_k=5_results.json", "r") as f:
    results11 = json.load(f)

with open("data/biencoder/default2/Qwen2.5-7B-Instruct_k=10_results.json", "r") as f:
    results12 = json.load(f)
    
    
###############################################################
with open("data/biencoder/default2/Qwen2.5-14B-Instruct_k=3_results.json", "r") as f:
    results13 = json.load(f)

with open("data/biencoder/default2/Qwen2.5-14B-Instruct_k=5_results.json", "r") as f:
    results14 = json.load(f)

with open("data/biencoder/default2/Qwen2.5-14B-Instruct_k=10_results.json", "r") as f:
    results15 = json.load(f)

In [33]:
score1 = scoring(results = results1, mention2gold = mention2gold)
print("data/default/Meta-Llama-3.1-8B-Instruct_k=3_results :", score1)
score2 = scoring(results = results2, mention2gold = mention2gold)
print("data/default/Meta-Llama-3.1-8B-Instruct_k=5_results:", score2)
score3 = scoring(results = results3, mention2gold = mention2gold)
print("data/default/Meta-Llama-3.1-8B-Instruct_k=10_results:", score3)

print("---------------------------------")

score4 = scoring(results = results4, mention2gold = mention2gold)
print("data/default/Mistral-7B-Instruct-v0.3_k=3_results :", score4)
score5 = scoring(results = results5, mention2gold = mention2gold)
print("data/default/Mistral-7B-Instruct-v0.3_k=5_results :", score5)
score6 = scoring(results = results6, mention2gold = mention2gold)
print("data/default/Mistral-7B-Instruct-v0.3_k=10_results :", score6)

print("---------------------------------")

score7 = scoring(results = results7, mention2gold = mention2gold)
print("data/default/Mistral-Nemo-Instruct-2407_k=3_results :", score7)
score8 = scoring(results = results8, mention2gold = mention2gold)
print("data/default/Mistral-Nemo-Instruct-2407_k=5_results:", score8)
score9 = scoring(results = results9, mention2gold = mention2gold)
print("data/default/Mistral-Nemo-Instruct-2407_k=10_results:", score9)

print("---------------------------------")

score10 = scoring(results = results10, mention2gold = mention2gold)
print("data/default/Qwen2.5-7B-Instruct_k=3_results :", score10)
score11 = scoring(results = results11, mention2gold = mention2gold)
print("data/default/Qwen2.5-7B-Instruct_k=5_results:", score11)
score12 = scoring(results = results12, mention2gold = mention2gold)
print("data/default/Qwen2.5-7B-Instruct_k=10_results:", score12)

print("---------------------------------")

score13 = scoring(results = results13, mention2gold = mention2gold)
print("data/default/Qwen2.5-14B-Instruct_k=3_results :", score13)
score14 = scoring(results = results14, mention2gold = mention2gold)
print("data/default/Qwen2.5-14B-Instruct_k=5_results:", score14)
score15 = scoring(results = results15, mention2gold = mention2gold)
print("data/default/Qwen2.5-14B-Instruct_k=10_results:", score15)



data/default/Meta-Llama-3.1-8B-Instruct_k=3_results : 0.8775
data/default/Meta-Llama-3.1-8B-Instruct_k=5_results: 0.8825
data/default/Meta-Llama-3.1-8B-Instruct_k=10_results: 0.89375
---------------------------------
data/default/Mistral-7B-Instruct-v0.3_k=3_results : 0.8275
data/default/Mistral-7B-Instruct-v0.3_k=5_results : 0.8275
data/default/Mistral-7B-Instruct-v0.3_k=10_results : 0.8375
---------------------------------
data/default/Mistral-Nemo-Instruct-2407_k=3_results : 0.87
data/default/Mistral-Nemo-Instruct-2407_k=5_results: 0.89125
data/default/Mistral-Nemo-Instruct-2407_k=10_results: 0.9
---------------------------------
data/default/Qwen2.5-7B-Instruct_k=3_results : 0.90625
data/default/Qwen2.5-7B-Instruct_k=5_results: 0.90875
data/default/Qwen2.5-7B-Instruct_k=10_results: 0.8975
---------------------------------
data/default/Qwen2.5-14B-Instruct_k=3_results : 0.8925
data/default/Qwen2.5-14B-Instruct_k=5_results: 0.9
data/default/Qwen2.5-14B-Instruct_k=10_results: 0.89125


In [34]:
def pooled_scoring(results, mention2gold):
    """
    Return the score of the model.
    -------
    results : list of dictionaries [{mention_id : {"predicted" : predicted CUI, "explanation" : explanation}}, ...]
    mention2gold : dictionary {mention_id : gold CUI or list of gold CUIs}
    """
    score = 0
    found_mentions = set()  # To track which mentions have already been correctly predicted

    # Iterate over each set of predictions (each result_i)
    for result in results:
        for mention_id, value in result.items():
            if mention_id in found_mentions:
                continue  # Skip if this mention was already counted
            
            predicted_cui = value["predicted"]
            gold_cuis = mention2gold[mention_id]  # Gold CUI or list of CUIs
            
            # Ensure gold_cuis is a list for consistent comparison
            if not isinstance(gold_cuis, list):
                gold_cuis = [gold_cuis]

            # Check if the predicted CUI is correct
            if predicted_cui in gold_cuis:
                score += 1  # Increment score only once per mention_id
                found_mentions.add(mention_id)  # Mark this mention as found

    return score / len(mention2gold)

In [77]:
results = [
    results1, 
    # results2, 
    # results3,    
    results4, 
    # results5, 
    # results6,
    results7, 
    # results8, 
    # results9,
    results10, 
    # results11, 
    # results12,
    results13, 
    # results14, 
    # results15,
    ]
score = pooled_scoring(results, mention2gold)
score

0.9473684210526315

In [ ]:
results_r = [
    results1r, 
    # results2r, 
    # results3r,    
    results4r, 
    # results5r, 
    # results6r,
    results7r, 
    # results8r, 
    # results9r,
    results10r, 
    # results11r, 
    # results12r,
    results13r, 
    # results14r, 
    # results15r,
    ]
score_r = pooled_scoring(results_r, mention2gold)
score_r

In [42]:
# len(results)
# len(mention2gold)

In [43]:
error_mentions = error_analysis(results = results2, 
                                ontology = ontology2, 
                                mention2gold = mention2gold, 
                                mention2context = mention2context)
error_mentions

defaultdict(dict,
            {'9288106.2': {'query_context': 'Clustering of missense mutations in the ataxia-telangiectasia gene in a [ENTITY_START] sporadic T-cell leukaemia. [ENTITY_END] Ataxia-telangiectasia (A-T) is a recessive multi-system disorder caused by mutations in the ATM gene at 11q22-q23 (ref. 3). The risk of cancer, especially lymphoid neoplasias, is substantially elevated in A-T',
              'gold_cui': {'MESH:D015458': {'cui': 'MESH:D015458',
                'name': 'Leukemia, T-Cell',
                'types': 'Disease',
                'aliases': 'Leukemia, Lymphocytic, T Cell|Leukemia, Lymphocytic, T-Cell|Leukemias, T-Cell|Leukemias, T-Cell Lymphocytic|Leukemias, T Lymphocytic|Leukemias, T-Lymphocytic|Leukemia, T Cell|Leukemia, T-Cell Lymphocytic|Leukemia, T Lymphocytic|Leukemia, T-Lymphocytic|Lymphocytic Leukemias, T|Lymphocytic Leukemias, T-Cell|Lymphocytic Leukemia, T|Lymphocytic Leukemia, T Cell|Lymphocytic Leukemia, T-Cell|T Cell Leukemia|T-Cell Leukemia|T-C

In [59]:
with open(
    "data/crossencoder/recall/Meta-Llama-3.1-8B-Instruct_k=3_results.json", "r"
) as f:
    results = json.load(f)

In [60]:
recall_r = recall_fn(results=results, mention2gold=mention2gold, ks = list(range(1,6)))
recall_r

{'recall@1': 0.8738396624472573,
 'recall@2': 0.9381856540084389,
 'recall@3': 0.9609704641350211,
 'recall@4': 0.9698312236286919,
 'recall@5': 0.9761603375527426}

In [61]:
unnormalized_recall = {
    f"unnormalized_{k}": v * number_hits_crossencoder[20] / 960
    for k, v in recall_r.items()
}
unnormalized_recall


{'unnormalized_recall@1': 0.731840717299578,
 'unnormalized_recall@2': 0.7857304852320676,
 'unnormalized_recall@3': 0.8048127637130802,
 'unnormalized_recall@4': 0.8122336497890295,
 'unnormalized_recall@5': 0.8175342827004218}

# vllm + aqlm


In [41]:
llm = LLM(
    model="ISTA-DASLab/Mixtral-8x7B-Instruct-v0_1-AQLM-2Bit-1x16-hf",
    enforce_eager=True,
)

tokenizer = llm.get_tokenizer()

WARNING 09-12 18:51:54 config.py:330] aqlm quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 09-12 18:51:54 config.py:378] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 09-12 18:51:54 llm_engine.py:213] Initializing an LLM engine (v0.6.0) with config: model='ISTA-DASLab/Mixtral-8x7B-Instruct-v0_1-AQLM-2Bit-1x16-hf', speculative_config=None, tokenizer='ISTA-DASLab/Mixtral-8x7B-Instruct-v0_1-AQLM-2Bit-1x16-hf', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=aqlm, enforce_eager=True, kv_cache_dtype=auto, quantization_param_path=None, devic

/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 09-12 18:52:04 model_runner.py:915] Starting to load model ISTA-DASLab/Mixtral-8x7B-Instruct-v0_1-AQLM-2Bit-1x16-hf...
INFO 09-12 18:52:04 selector.py:217] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 09-12 18:52:04 selector.py:116] Using XFormers backend.
INFO 09-12 18:52:04 weight_utils.py:236] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:03<00:06,  3.10s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:06<00:03,  3.53s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:09<00:00,  3.32s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:09<00:00,  3.33s/it]



INFO 09-12 18:52:16 model_runner.py:926] Loading model weights took 12.2120 GB
INFO 09-12 18:52:45 gpu_executor.py:122] # GPU blocks: 11939, # CPU blocks: 2048


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [30]:
# def prompt_vllm_aqlm(mention, context, system_instructions, candidates, topk_examples, llm, tokenizer, sampling_params):
#     ''' 
#     mention : str (name of the mention to be linked)
#     context : str (context where the mention appears)
#     system_instructions : str (instructions for the LLM)
#     candidates : list of list of CUIs : [[cui1], [cui2, cui3], ...]
#     topk_examples : str (top k examples of similar contexts)
#     llm : LLM model
#     tokenizer : AutoTokenizer
#     sampling_params : SamplingParams config
#     '''
    
#     prompt_text = f"""
#     System Instructions: {system_instructions} \n
    
#     Here are a few examples: \n
#     {topk_examples} \n

#     This is the specific mention that needs to be linked to the correct entity: {mention} \n

#     This is the context where the mention appears: \n
#     {context} \n
    
#     These are the candidate entities to choose from: \n
#     {candidates} \n
    
#     You MUST PROVIDE an ANSWER among the candidates. \n

#     Return the answer in the following format: CUI
#     For instance : "MESH:D000000" "OMIM:000000" are valid answers. \n
#     Do not add provide any explanations ! But you MUST give ONE answer.
#     """
#     conversations = tokenizer.apply_chat_template(
#         [{'role': 'user', 'content': prompt_text}],
#         tokenize=False,
#     )

#     # Decode the generated tokens into text
#     outputs = llm.generate([conversations], sampling_params=sampling_params, use_tqdm=False)
#     answer = outputs[0].outputs[0].text

#     return answer


In [43]:
mention = "9931324.1"
text = mention2text[mention]
context = mention2context[mention]
candidates = get_candidates_data(mention2biencoder_candidates[mention], ontology2)
topk = topk_examples(model = model,
                    index = index,
                    query=context, 
                    corpus=corpus, 
                    TrainMap_context2mention=TrainMap_context2mention, 
                    train_mention2text=train_mention2text, 
                    train_mention2gold=train_mention2gold, 
                    ontology=ontology2, 
                    k=3)

pred_cui = prompt_vllm_aqlm(mention=text, 
                       context=context, 
                       system_instructions=system_instructions,
                       candidates=candidates, 
                       topk_examples=topk,
                       llm=llm,
                       tokenizer=tokenizer,
                       sampling_params=sampling_params)

# Output the results
print("Mention ID:", mention)
print("Context:", context)
print("Top k examples:", topk)
print("Text:", mention2text[mention])
print("Gold CUI:", mention2gold[mention])
print("Predicted CUI:", pred_cui)

Batches: 100%|██████████| 1/1 [00:00<00:00, 36.35it/s]


(1, 768)
Mention ID: 9931324.1
Context: Missense mutations in the most ancient residues of the PAX6 paired domain underlie a spectrum of human [ENTITY_START] congenital eye malformations. [ENTITY_END] Mutations of the human PAX6 gene underlie aniridia (congenital absence of the iris), a rare dominant malformation of the eye. The spectrum of PAX6 mutations in aniridia patients is highly
Top k examples: Mention 1: aniridia || Context: Three novel [ENTITY_START] aniridia [ENTITY_END] mutations in the human PAX6 gene. Aniridia (iris hypoplasia) is an autosomal dominant congenital disorder of the eye. Mutations in the human aniridia (PAX6) gene have now been identified in many || Correct CUI: {'MESH:D015783': {'cui': 'MESH:D015783', 'name': 'Aniridia', 'types': 'Disease', 'aliases': 'Absent Iris|AN|AN1|AN2|AN3|ANIRIDIA 1|ANIRIDIA 2|ANIRIDIA 3|ANIRIDIA II, FORMERLY;AN2, FORMERLY CATARACT, CONGENITAL, WITH LATE-ONSET CORNEAL DYSTROPHY, INCLUDED|Congenital Aniridia|Irideremia', 'definition': '

In [44]:
# def evaluate_vllm_aqlm(
#     llm,
#     nlp_model,
#     tokenizer,
#     index,
#     system_instructions,
#     mentions,
#     ontology,
#     corpus,
#     mention2context,
#     mention2biencoder_candidates,
#     mention2text,
#     TrainMap_context2mention,
#     train_mention2text,
#     train_mention2gold,
#     k,
#     sampling_params,
# ):
#     """
#     Run "prompt" function for each mention in the list of mentions.
#     Returns a dictionary {mention_id : predicted CUI}
#     -------
#     llm : LLM model
#     nlp_model : SentenceTransformer model
#     tokenizer : AutoTokenizer
#     index : faiss index
#     system_instructions : str (instructions for the LLM)
#     mentions : list (mention_ids)
#     ontology : BiomedicalOntology object
#     corpus : list of str (all context sentences)
#     mention2context : dict (mention_id : context)
#     mention2biencoder_candidates : dict (mention_id : list of candidate CUIs)
#     mention2text : dict (mention_id : mention name)
#     TrainMap_context2mention : dict (context sentence to mention_id)
#     train_mention2text : dict (mention_id to mention name)
#     train_mention2gold : dict (mention_id to gold CUI)
#     k : int (number of nearest neighbors)
#     sampling_params : SamplingParams config
#     """
#     results = {}
#     for i in range(len(mentions)) :
#         mention_id = mentions[i]
#         mention_name = mention2text[mention_id]
#         context = mention2context[mention_id]
#         candidates = get_candidates_data(mention2biencoder_candidates[mentions[i]], ontology)
#         # candidates = get_candidates_data_v2(mention2crossencoder_candidates[mention])
#         topk = topk_examples(
#             model=nlp_model,  # sentence transformer model
#             index=index,
#             query=context,
#             corpus=corpus,
#             TrainMap_context2mention=TrainMap_context2mention,
#             train_mention2text=train_mention2text,
#             train_mention2gold=train_mention2gold,
#             ontology=ontology,
#             k=k,
#         )
#         text = prompt_vllm_aqlm(
#             mention=mention_name,
#             context=context,
#             system_instructions=system_instructions,
#             candidates=candidates,
#             topk_examples=topk,
#             llm=llm,
#             tokenizer=tokenizer,
#             sampling_params=sampling_params,
#         )
#         print("mention ID :", mention_id , "|| LLM answer :", text)
#         cand = extract_cui(text)
#         results[mention_id] = cand
#         if i%20 == 0:
#             print(f"i = {i}")

#     return results
    


In [46]:
results = evaluate_vllm_aqlm(llm=llm,
                        nlp_model=model,
                        tokenizer=tokenizer,
                        index=index,
                        system_instructions=system_instructions,
                        mentions=mentions[:5],
                        ontology=ontology2,
                        corpus=corpus,
                        mention2context=mention2context,
                        mention2biencoder_candidates=mention2biencoder_candidates,
                        mention2text=mention2text,
                        TrainMap_context2mention=TrainMap_context2mention,
                        train_mention2text=train_mention2text,
                        train_mention2gold=train_mention2gold,
                        k=10,
                        sampling_params=sampling_params)
results

Batches: 100%|██████████| 1/1 [00:00<00:00, 97.73it/s]


(1, 768)
mention ID : 9288106.1 || LLM answer :  MESH:D001260
i = 0


Batches: 100%|██████████| 1/1 [00:00<00:00, 87.08it/s]

(1, 768)


mention ID : 9288106.2 || LLM answer :  'MESH:D016399'


...

mention ID : 9288106.4 || LLM answer :  MESH:D001260


Batches: 100%|██████████| 1/1 [00:00<00:00, 107.52it/s]

(1, 768)


mention ID : 9288106.5 || LLM answer :  MESH:D030342


{'9288106.1': 'MESH:D001260',
 '9288106.2': 'MESH:D016399',
 '9288106.3': 'MESH:D001260',
 '9288106.4': 'MESH:D001260',
 '9288106.5': 'MESH:D030342'}

In [47]:
results

{'9288106.1': 'MESH:D001260',
 '9288106.2': 'MESH:D016399',
 '9288106.3': 'MESH:D001260',
 '9288106.4': 'MESH:D001260',
 '9288106.5': 'MESH:D030342'}

In [49]:
score = scoring(results=results, mention2gold=mention2gold)
score

0.8

### Fine-tuning

In [37]:
from datasets import load_metric

bleu = load_metric("bleu")

prediction1 = [["there", "is", "a", "need", "for", "adequate", "and", "predictable", "resources"]]
prediction2 = [["resources", "be", "sufficient", "and", "predictable", "to"]] 

reference1 = [[["resources", "have", "to", "be", "sufficient", "and", "they", "have", "to", "be", "predictable"]]]
reference2 = [[["adequate", "and", "predictable", "resources", "are", "required"]]]

bleu_c1r1 = bleu.compute(predictions=prediction1, references=reference1, max_order=2)
print("bleu score for c1 using r1 :", bleu_c1r1)
bleu_c1r2 = bleu.compute(predictions=prediction1, references=reference2, max_order=2)
print("bleu score for c1 using r2 :", bleu_c1r2)

bleu_c2r1 = bleu.compute(predictions=prediction2, references=reference1, max_order=2)
print("bleu score for c2 using r1 :", bleu_c2r1)
bleu_c2r2 = bleu.compute(predictions=prediction2, references=reference1, max_order=2)
print("bleu score for c2 using r2 :", bleu_c2r2)

/nethome/cye73/conda_envs/bioel/lib/python3.9/site-packages/datasets/load.py:756: FutureWarning: The repository for bleu contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/bleu/bleu.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(


bleu score for p1 using r1 : {'bleu': 0.0, 'precisions': [0.3333333333333333, 0.0], 'brevity_penalty': 0.8007374029168082, 'length_ratio': 0.8181818181818182, 'translation_length': 9, 'reference_length': 11}
bleu score for p1 using r2 : {'bleu': 0.408248290463863, 'precisions': [0.4444444444444444, 0.375], 'brevity_penalty': 1.0, 'length_ratio': 1.5, 'translation_length': 9, 'reference_length': 6}
bleu score for p2 using r1 : {'bleu': 0.2748640411822265, 'precisions': [1.0, 0.4], 'brevity_penalty': 0.43459820850707814, 'length_ratio': 0.5454545454545454, 'translation_length': 6, 'reference_length': 11}
bleu score for p2 using r2 : {'bleu': 0.2748640411822265, 'precisions': [1.0, 0.4], 'brevity_penalty': 0.43459820850707814, 'length_ratio': 0.5454545454545454, 'translation_length': 6, 'reference_length': 11}


In [38]:
prediction1 = [["there", "is", "a", "need", "for", "adequate", "and", "predictable", "resources"]]
prediction2 = [["resources", "be", "sufficient", "and", "predictable", "to"]] 

reference1 = [[["resources", "have", "to", "be", "sufficient", "and", "they", "have", "to", "be", "predictable"]], [["adequate", "and", "predictable", "resources", "are", "required"]]]
# reference2 = [[["adequate", "and", "predictable", "resources", "are", "required"]]]

bleu_c1r1 = bleu.compute(predictions=prediction1, references=reference1, max_order=2)
print("bleu score for c1 using r1 :", bleu_c1r1)


bleu_c2r1 = bleu.compute(predictions=prediction2, references=reference1, max_order=2)
print("bleu score for c2 using r1 :", bleu_c2r1)

ValueError: Mismatch in the number of predictions (1) and references (2)